In [1]:
from dask import delayed, compute
import pandas as pd
import zarr
from pathlib import Path

from src.data_types.ZarrToIndices import compute_indices_for_zarr
from src.data_types.NetcdfToIndices import compute_indices_for_netcdf
from src.data_types.ASCIIToIndices import compute_indices_for_cls

In [2]:
import pandas as pd
import numpy as np

def results_to_dataframe(results):
    """
    Convert nested results structure into a tidy pandas DataFrame.
    Works for multiple methods (metpy, sharppy, etc.).
    """
    flat_records = []

    for sounding in results:
        if sounding:  # skip empty lists
            for record in sounding:
                clean_record = {}

                for key, value in record.items():
                    # Convert numpy scalars to native Python
                    if isinstance(value, np.generic):
                        clean_record[key] = value.item()
                    else:
                        clean_record[key] = value

                flat_records.append(clean_record)

    df = pd.DataFrame(flat_records)

    # Optional: sort for readability
    if "datetime" in df.columns:
        df = df.sort_values(["datetime", "method"]).reset_index(drop=True)

    return df

### Indices from Zarr Store

In [10]:
%%time

zarr_store = "/lustre/desc1/scratch/myasears/sounding_data/zarr/m2hats"
group_keys = list(zarr.open(zarr_store, mode="r").group_keys())

tasks = [delayed(compute_indices_for_zarr)(zarr_store, key) for key in group_keys]
results = compute(*tasks)
zarr_df = results_to_dataframe(results)

CPU times: user 3.85 s, sys: 265 ms, total: 4.12 s
Wall time: 3.58 s


In [14]:
def get_dir_size(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
zarr_store = "/lustre/desc1/scratch/myasears/sounding_data/zarr/m2hats"
zarr_size = get_dir_size(zarr_store)
print(f"Zarr size: {zarr_size / 1e6:.2f} MB")

Zarr size: 32.64 MB


### Indices from NetCDF files

In [11]:
%%time

netcdf_files = sorted(Path("/lustre/desc1/scratch/myasears/sounding_data/m2hats").glob("*.nc"))

tasks = [delayed(compute_indices_for_netcdf)(fp) for fp in netcdf_files]
results = compute(*tasks)
netcdf_df = results_to_dataframe(results)

CPU times: user 4.36 s, sys: 193 ms, total: 4.55 s
Wall time: 3.99 s


In [15]:
netcdf_files = sorted(Path("/lustre/desc1/scratch/myasears/sounding_data/m2hats").glob("*.nc"))
netcdf_size = sum(f.stat().st_size for f in netcdf_files)
print(f"NetCDF total size: {netcdf_size / 1e6:.2f} MB")

NetCDF total size: 52.73 MB


### Indices from CLS (ASCII) files

In [ ]:
cls_files = sorted(Path("data").glob("*.cls"))

tasks = [delayed(compute_indices_for_cls)(fp) for fp in cls_files]
results = compute(*tasks)
ascii_df = results_to_dataframe(results)